In [5]:
"""
=============================================================================
SCRIPT 3: Session Visualization
=============================================================================
Outputs:
  session_overview.png        — Full session timeline (all 14 trials)
  trial_detail_block1.png     — Zoomed trial detail for span-2 block
  trial_detail_block7.png     — Zoomed trial detail for span-5 block

What is shown:
  Panel 1: Expected stimulation spike train (0.1 mA pulses keyed to task events)
  Panel 2: Neural LFP amplitude (250 Hz, Left ATN, downsampled for display)
  Panel 3: LFP power envelope (2 Hz, Left ATN)
  Event markers: fixation, stimulus, response, feedback for every sub-trial

IMPORTANT TIMING NOTE:
  The neural (250 Hz) and LFP-power (2 Hz) recordings started at E-Prime
  time ~348,000 ms — approximately 7,500 ms AFTER the task ended (~340,514 ms).
  There is therefore NO neural overlap with the task in this session.

  The stimulation spike train is RECONSTRUCTED from E-Prime event times:
  each task event (fixation onset, stimulus onset, response onset, feedback onset)
  had a 0.1 mA spike delivered just before it by protocol. We plot these as
  vertical lines on the E-Prime timeline so the full picture is visible.

  The neural data section shows the post-task resting-state LFP for reference.
=============================================================================
"""

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
import warnings
warnings.filterwarnings('ignore')

# ── CONFIG ────────────────────────────────────────────────────────────────────
EVENTS_PATH = r"C:\Users\ASSUS\ATN\Digit Span Backwards\Preprocessed Data\events_data_processed.csv"
NEURAL_PATH = r"C:\Users\ASSUS\ATN\Digit Span Backwards\Preprocessed Data\neural_data_processed.csv"
LFP_PATH    = r"C:\Users\ASSUS\ATN\Digit Span Backwards\Preprocessed Data\neural_lfp_power_processed.csv"
PLOTS_DIR   = r"C:\Users\ASSUS\ATN\Digit Span Backwards\Plots"

import os
os.makedirs(PLOTS_DIR, exist_ok=True)

# Stimulation pulse appears just BEFORE each event onset (protocol design)
SPIKE_LEAD_MS = 50   # spike delivered 50ms before event onset

# Neural display: downsample 250Hz to every Nth sample for speed
DS_FACTOR = 5   # 250Hz -> 50Hz for display (still crisp)

# Colour palette — publication-friendly, colour-blind safe
C_FIX      = '#2196F3'   # blue      — fixation
C_STIM     = '#FF9800'   # orange    — stimulus digit
C_RESP     = '#9C27B0'   # purple    — response
C_FEED_OK  = '#4CAF50'   # green     — correct feedback
C_FEED_ERR = '#F44336'   # red       — incorrect feedback
C_SPIKE    = '#E91E63'   # pink-red  — stimulation spike
C_NEURAL_L = '#1565C0'   # dark blue — neural Left
C_NEURAL_R = '#0288D1'   # light blue— neural Right (not used in main plot)
C_LFP      = '#00897B'   # teal      — LFP power
C_SHADE    = '#ECEFF1'   # very light grey — trial shading

ALPHA_BAND  = 0.10
ALPHA_EVLINE= 0.55
ALPHA_SPIKE = 0.90

# ── LOAD DATA ─────────────────────────────────────────────────────────────────
print("Loading data...")
events = pd.read_csv(EVENTS_PATH)
neural = pd.read_csv(NEURAL_PATH)
lfp    = pd.read_csv(LFP_PATH)

left_neural = neural[neural['hemisphere'] == 'Left'].copy().reset_index(drop=True)
left_neural_ds = left_neural.iloc[::DS_FACTOR].copy()

print(f"  Events    : {len(events)} sub-trials, {events['block'].nunique()} blocks")
print(f"  Neural    : {len(left_neural):,} samples (Left, 250Hz)")
print(f"  LFP power : {len(lfp)} samples (2Hz)")

# ── BUILD STIMULATION SPIKE TRAIN ─────────────────────────────────────────────
# Per protocol: 0.1 mA spike delivered just before:
#   - Each fixation onset
#   - Each stimulus onset
#   - Response screen onset (once per trial)
#   - Feedback onset (once per trial)
# We mark spike time = event_onset - SPIKE_LEAD_MS

spike_events = []

for _, row in events.iterrows():
    blk   = int(row['block'])
    trl   = int(row['trial'])
    sub   = int(row['sub_trial'])
    span  = int(row['span_size'])
    acc   = int(row['is_correct'])

    # Spike before fixation (every sub-trial)
    spike_events.append({
        'time_ms'   : row['fixation_onset_ms'] - SPIKE_LEAD_MS,
        'event_type': 'pre_fixation',
        'block': blk, 'trial': trl, 'sub_trial': sub,
        'span_size': span, 'is_correct': acc
    })

    # Spike before stimulus (every sub-trial)
    spike_events.append({
        'time_ms'   : row['stimulus_onset_ms'] - SPIKE_LEAD_MS,
        'event_type': 'pre_stimulus',
        'block': blk, 'trial': trl, 'sub_trial': sub,
        'span_size': span, 'is_correct': acc
    })

    # Spike before response (only on last sub-trial of each trial)
    if sub == span:
        spike_events.append({
            'time_ms'   : row['response_onset_ms'] - SPIKE_LEAD_MS,
            'event_type': 'pre_response',
            'block': blk, 'trial': trl, 'sub_trial': sub,
            'span_size': span, 'is_correct': acc
        })
        # Spike before feedback
        spike_events.append({
            'time_ms'   : row['feedback_onset_ms'] - SPIKE_LEAD_MS,
            'event_type': 'pre_feedback',
            'block': blk, 'trial': trl, 'sub_trial': sub,
            'span_size': span, 'is_correct': acc
        })

df_spikes = pd.DataFrame(spike_events).sort_values('time_ms').reset_index(drop=True)
print(f"  Spike events generated: {len(df_spikes)}")

# ── HELPER: draw event marker lines ──────────────────────────────────────────
def draw_event_markers(ax, events_df, ymin=0, ymax=1, alpha=ALPHA_EVLINE, lw=1.0):
    """Draw vertical lines for each event type."""
    for _, row in events_df.iterrows():
        sub  = int(row['sub_trial'])
        span = int(row['span_size'])
        acc  = int(row['is_correct'])

        # Fixation
        ax.axvline(row['fixation_onset_ms']/1000, color=C_FIX,
                   alpha=alpha, lw=lw, ls='--')
        # Stimulus
        ax.axvline(row['stimulus_onset_ms']/1000, color=C_STIM,
                   alpha=alpha, lw=lw, ls='-')
        # Response (only on last sub-trial)
        if sub == span:
            ax.axvline(row['response_onset_ms']/1000, color=C_RESP,
                       alpha=alpha+0.2, lw=lw*1.3, ls='-.')
            fc = C_FEED_OK if acc == 1 else C_FEED_ERR
            ax.axvline(row['feedback_onset_ms']/1000, color=fc,
                       alpha=alpha+0.2, lw=lw*1.3, ls=':')

def shade_trials(ax, events_df, ymin, ymax):
    """Alternating light shading per trial."""
    for (blk, trl), grp in events_df.groupby(['block', 'trial']):
        t_start = grp['fixation_onset_ms'].min() / 1000
        t_end   = grp['feedback_offset_ms'].max() / 1000
        if (blk + trl) % 2 == 0:
            ax.axvspan(t_start, t_end, color=C_SHADE, alpha=0.5, zorder=0)

# ═══════════════════════════════════════════════════════════════════════════════
# FIGURE 1: Full session overview
# ═══════════════════════════════════════════════════════════════════════════════
print("\nGenerating Figure 1: Full session overview...")

fig, axes = plt.subplots(
    4, 1,
    figsize=(28, 14),
    gridspec_kw={'height_ratios': [1.2, 0.8, 1.8, 1.4]},
    sharex=False
)
fig.patch.set_facecolor('white')
fig.suptitle(
    'DBS Digit Span Backward — Full Session Overview (Subject 6, 2026-03-05)',
    fontsize=15, fontweight='bold', y=0.98
)

# ── PANEL 0: Stimulation spike train (task time) ──────────────────────────────
ax0 = axes[0]
ax0.set_facecolor('white')

spike_colors = {
    'pre_fixation' : C_FIX,
    'pre_stimulus' : C_STIM,
    'pre_response' : C_RESP,
    'pre_feedback' : C_FEED_OK,
}

task_t_start = events['fixation_onset_ms'].min() / 1000
task_t_end   = events['feedback_offset_ms'].max() / 1000

shade_trials(ax0, events, 0, 1)
draw_event_markers(ax0, events, alpha=0.12, lw=0.6)

# Plot spikes as vertical stems
for _, sp in df_spikes.iterrows():
    t   = sp['time_ms'] / 1000
    col = spike_colors.get(sp['event_type'], C_SPIKE)
    ax0.annotate('', xy=(t, 0.85), xytext=(t, 0.05),
                 arrowprops=dict(arrowstyle='->', color=col, lw=1.5))

ax0.set_xlim(task_t_start - 1, task_t_end + 1)
ax0.set_ylim(0, 1)
ax0.set_yticks([0.05, 0.85])
ax0.set_yticklabels(['0', '0.1 mA'], fontsize=9)
ax0.set_ylabel('Stim\n(mA)', fontsize=10, fontweight='bold')
ax0.set_title('Panel 1 — Stimulation Spike Train (reconstructed from E-Prime event times, 0.1 mA per spike)',
              fontsize=10, loc='left', pad=3)

# Block labels at top
for blk, grp in events.groupby('block'):
    mid = (grp['fixation_onset_ms'].min() + grp['feedback_offset_ms'].max()) / 2 / 1000
    span = grp['span_size'].iloc[0]
    ax0.text(mid, 0.95, f'B{blk}\n({span}d)', ha='center', va='top',
             fontsize=7.5, color='#37474F', fontweight='bold')

# ── PANEL 1: Trial accuracy bar ────────────────────────────────────────────────
ax1 = axes[1]
ax1.set_facecolor('white')

trial_level = events.groupby(['block','trial']).agg(
    t_start=('fixation_onset_ms','min'),
    t_end=('feedback_offset_ms','max'),
    acc=('is_correct','first'),
    span=('span_size','first')
).reset_index()

for _, row in trial_level.iterrows():
    col = C_FEED_OK if row['acc'] == 1 else C_FEED_ERR
    ax1.barh(0, (row['t_end'] - row['t_start'])/1000,
             left=row['t_start']/1000, height=0.6,
             color=col, alpha=0.75, edgecolor='white', lw=0.5)
    mid = (row['t_start'] + row['t_end']) / 2 / 1000
    label = '✓' if row['acc'] == 1 else '✗'
    ax1.text(mid, 0, label, ha='center', va='center',
             fontsize=11, color='white', fontweight='bold')

ax1.set_xlim(task_t_start - 1, task_t_end + 1)
ax1.set_ylim(-0.5, 0.5)
ax1.set_yticks([])
ax1.set_ylabel('Accuracy', fontsize=10, fontweight='bold')
ax1.set_title('Panel 2 — Trial Accuracy  (green=correct, red=incorrect)',
              fontsize=10, loc='left', pad=3)

# Span size labels
for _, row in trial_level.iterrows():
    mid = (row['t_start'] + row['t_end']) / 2 / 1000
    ax1.text(mid, 0.38, f"{row['span']}d", ha='center', va='bottom',
             fontsize=7.5, color='#37474F')

# ── PANEL 2: Neural LFP (250Hz, post-task, Left ATN) ─────────────────────────
ax2 = axes[2]
ax2.set_facecolor('#FAFAFA')

neural_t = left_neural_ds['eprime_time_ms'].values / 1000
neural_v = left_neural_ds['amplitude_uv'].values

ax2.plot(neural_t, neural_v, color=C_NEURAL_L, lw=0.4, alpha=0.85,
         label='Left ATN (250Hz, ds×5)')

# Shade neural recording window
ax2.axvspan(neural_t[0], neural_t[-1], color='#E3F2FD', alpha=0.3, zorder=0,
            label='Neural recording window')

# Mark task end
ax2.axvline(task_t_end, color='#F44336', lw=2, ls='--', alpha=0.8,
            label=f'Task ended ({task_t_end:.1f}s)')

ax2.set_ylabel('LFP Amplitude\n(µV)', fontsize=10, fontweight='bold')
ax2.set_title('Panel 3 — Raw LFP Signal, Left ATN (250Hz, ZERO_THREE_LEFT)  '
              '[ Recording starts ~7.5s after task ended ]',
              fontsize=10, loc='left', pad=3)
ax2.legend(fontsize=8, loc='upper right', framealpha=0.8)
ax2.grid(axis='y', alpha=0.3, lw=0.5)

# ── PANEL 3: LFP power + stim mA (2Hz) ────────────────────────────────────────
ax3  = axes[3]
ax3b = ax3.twinx()
ax3.set_facecolor('#FAFAFA')

lfp_t    = lfp['eprime_time_ms'].values / 1000
lfp_pow  = lfp['lfp_power_left'].values
stim_ma  = lfp['stim_ma_left'].values

ax3.fill_between(lfp_t, lfp_pow, alpha=0.35, color=C_LFP)
ax3.plot(lfp_t, lfp_pow, color=C_LFP, lw=1.2, label='LFP power Left (2Hz)')

ax3b.plot(lfp_t, stim_ma, color=C_SPIKE, lw=1.5, alpha=0.8,
          drawstyle='steps-post', label='Stim mA Left')
ax3b.set_ylabel('Stim amplitude (mA)', fontsize=9, color=C_SPIKE)
ax3b.tick_params(axis='y', colors=C_SPIKE)

ax3.axvline(task_t_end, color='#F44336', lw=2, ls='--', alpha=0.8)
ax3.set_ylabel('LFP Power\n(device units)', fontsize=10, fontweight='bold')
ax3.set_title('Panel 4 — LFP Power Envelope (2Hz) + Stimulation Amplitude (mA), Left ATN',
              fontsize=10, loc='left', pad=3)
ax3.grid(axis='y', alpha=0.3, lw=0.5)

lines1, labels1 = ax3.get_legend_handles_labels()
lines2, labels2 = ax3b.get_legend_handles_labels()
ax3.legend(lines1 + lines2, labels1 + labels2, fontsize=8,
           loc='upper right', framealpha=0.8)

# ── Shared x-axis setup ───────────────────────────────────────────────────────
# Panels 0,1 show task window; panels 2,3 show neural window
neural_x_start = neural_t[0] - 5
neural_x_end   = neural_t[-1] + 5

for ax in [ax2, ax3]:
    ax.set_xlim(neural_x_start, neural_x_end)
    ax.set_xlabel('E-Prime Time (s)', fontsize=10)

for ax in [ax0, ax1]:
    ax.set_xlabel('E-Prime Time (s)', fontsize=10)
    ax.tick_params(axis='x', labelsize=9)

for ax in axes:
    ax.tick_params(axis='x', labelsize=9)
    ax.tick_params(axis='y', labelsize=9)
    for spine in ['top', 'right']:
        ax.spines[spine].set_visible(False)

# ── Legend for event types ────────────────────────────────────────────────────
legend_elements = [
    Line2D([0],[0], color=C_FIX,      lw=2, ls='--', label='Fixation onset'),
    Line2D([0],[0], color=C_STIM,     lw=2, ls='-',  label='Stimulus (digit) onset'),
    Line2D([0],[0], color=C_RESP,     lw=2, ls='-.', label='Response screen onset'),
    Line2D([0],[0], color=C_FEED_OK,  lw=2, ls=':',  label='Feedback onset (correct)'),
    Line2D([0],[0], color=C_FEED_ERR, lw=2, ls=':',  label='Feedback onset (incorrect)'),
    mpatches.Patch(color=C_SPIKE, alpha=0.8, label='Stimulation spike (0.1 mA)'),
]
fig.legend(handles=legend_elements, loc='lower center', ncol=6,
           fontsize=9, framealpha=0.9,
           bbox_to_anchor=(0.5, -0.01), bbox_transform=fig.transFigure)

plt.tight_layout(rect=[0, 0.04, 1, 0.98], h_pad=1.5)
out_path_overview = os.path.join(PLOTS_DIR, 'session_overview.png')
plt.savefig(out_path_overview, dpi=160, bbox_inches='tight', facecolor='white')
plt.show()
print(f"  ✓ session_overview.png saved → {out_path_overview}")

# ═══════════════════════════════════════════════════════════════════════════════
# FIGURE 2: Detailed trial view — one trial at a time, all blocks
# ═══════════════════════════════════════════════════════════════════════════════
def plot_trial_detail(block_num, title_suffix, out_path):
    """Zoomed plot of a single block (2 trials) with full sub-trial detail."""
    ev_blk = events[events['block'] == block_num].copy()
    sp_blk = df_spikes[df_spikes['block'] == block_num].copy()

    t_start_s = (ev_blk['fixation_onset_ms'].min() - 500) / 1000
    t_end_s   = (ev_blk['feedback_offset_ms'].max() + 500) / 1000

    fig2, axes2 = plt.subplots(2, 1, figsize=(18, 7),
                               gridspec_kw={'height_ratios': [1, 0.5]})
    fig2.patch.set_facecolor('white')
    fig2.suptitle(
        f'Block {block_num} — {title_suffix}',
        fontsize=13, fontweight='bold'
    )

    ax_s = axes2[0]
    ax_a = axes2[1]

    # ── Stim spike panel ──
    ax_s.set_facecolor('white')
    shade_trials(ax_s, ev_blk, 0, 1)

    # Full event marker lines
    draw_event_markers(ax_s, ev_blk, alpha=0.35, lw=1.2)

    # Spike arrows
    for _, sp in sp_blk.iterrows():
        t   = sp['time_ms'] / 1000
        col = spike_colors.get(sp['event_type'], C_SPIKE)
        ax_s.annotate('', xy=(t, 0.82), xytext=(t, 0.05),
                      arrowprops=dict(arrowstyle='->', color=col, lw=2.0))

    # Label each sub-trial event
    for _, row in ev_blk.iterrows():
        sub  = int(row['sub_trial'])
        digit = int(row['digit_shown'])
        t_fix = row['fixation_onset_ms'] / 1000
        t_stim = row['stimulus_onset_ms'] / 1000
        ax_s.text(t_fix, 0.93,  f'Fix\n(S{sub})', ha='center', va='top',
                  fontsize=7, color=C_FIX, fontweight='bold')
        ax_s.text(t_stim, 0.93, f'"{digit}"\n(S{sub})', ha='center', va='top',
                  fontsize=7, color=C_STIM, fontweight='bold')
        if sub == int(row['span_size']):
            t_resp = row['response_onset_ms'] / 1000
            t_feed = row['feedback_onset_ms'] / 1000
            acc    = int(row['is_correct'])
            ax_s.text(t_resp, 0.93, 'Resp', ha='center', va='top',
                      fontsize=7, color=C_RESP, fontweight='bold')
            fc = C_FEED_OK if acc else C_FEED_ERR
            ax_s.text(t_feed, 0.93, '✓' if acc else '✗', ha='center', va='top',
                      fontsize=10, color=fc, fontweight='bold')

    ax_s.set_xlim(t_start_s, t_end_s)
    ax_s.set_ylim(0, 1.05)
    ax_s.set_yticks([0.05, 0.82])
    ax_s.set_yticklabels(['0 mA', '0.1 mA'], fontsize=9)
    ax_s.set_ylabel('Stimulation', fontsize=10, fontweight='bold')
    ax_s.set_title('Stimulation spike train  (arrows) + Event markers (lines)',
                   fontsize=9, loc='left')

    # Trial divider
    trial_change = ev_blk[ev_blk['sub_trial'] == 1]
    for _, row in trial_change.iterrows():
        trl = int(row['trial'])
        span = int(row['span_size'])
        t_tc = row['fixation_onset_ms'] / 1000
        ax_s.axvline(t_tc, color='#607D8B', lw=2, alpha=0.4)
        ax_s.text(t_tc + 0.05, 0.08, f'Trial {trl}\n(span {span})',
                  fontsize=8, color='#37474F', va='bottom')

    for spine in ['top', 'right']:
        ax_s.spines[spine].set_visible(False)

    # ── Accuracy / digit sequence panel ──
    ax_a.set_facecolor('white')
    shade_trials(ax_a, ev_blk, 0, 1)

    for (blk, trl), grp in ev_blk.groupby(['block', 'trial']):
        digits   = grp['digit_shown'].tolist()
        reversed_str = ''.join(str(d) for d in reversed(digits))
        resp_str = str(grp['response_made'].iloc[0])
        acc      = int(grp['is_correct'].iloc[0])
        mid_t    = (grp['fixation_onset_ms'].min() + grp['feedback_offset_ms'].max()) / 2 / 1000
        col      = C_FEED_OK if acc else C_FEED_ERR

        shown_str = ' → '.join(str(d) for d in digits)
        label_str = f'Shown: {shown_str}\nCorrect↩: {reversed_str}   Response: {resp_str}'
        ax_a.text(mid_t, 0.55, label_str, ha='center', va='center',
                  fontsize=9, color='#37474F',
                  bbox=dict(boxstyle='round,pad=0.3', facecolor=col,
                            alpha=0.15, edgecolor=col, lw=1.2))

    ax_a.set_xlim(t_start_s, t_end_s)
    ax_a.set_ylim(0, 1)
    ax_a.set_yticks([])
    ax_a.set_xlabel('E-Prime Time (s)', fontsize=10)
    ax_a.set_ylabel('Digit seq', fontsize=10, fontweight='bold')
    ax_a.set_title('Digit sequences shown and patient responses', fontsize=9, loc='left')
    for spine in ['top', 'right', 'left']:
        ax_a.spines[spine].set_visible(False)

    # Shared legend
    legend_elements2 = [
        Line2D([0],[0], color=C_FIX,      lw=1.5, ls='--', label='Fixation'),
        Line2D([0],[0], color=C_STIM,     lw=1.5, ls='-',  label='Stimulus'),
        Line2D([0],[0], color=C_RESP,     lw=1.5, ls='-.', label='Response'),
        Line2D([0],[0], color=C_FEED_OK,  lw=1.5, ls=':',  label='Feedback (correct)'),
        Line2D([0],[0], color=C_FEED_ERR, lw=1.5, ls=':',  label='Feedback (incorrect)'),
    ]
    fig2.legend(handles=legend_elements2, loc='lower center', ncol=5,
                fontsize=9, framealpha=0.9, bbox_to_anchor=(0.5, -0.03))

    plt.tight_layout(rect=[0, 0.05, 1, 0.97], h_pad=1.5)
    full_out_path = os.path.join(PLOTS_DIR, out_path)
    plt.savefig(full_out_path, dpi=160, bbox_inches='tight', facecolor='white')
    plt.show()
    print(f"  ✓ {out_path} saved → {full_out_path}")

# Generate per-block detail plots
for blk in sorted(events['block'].unique()):
    span = events[events['block'] == blk]['span_size'].iloc[0]
    acc_vals = events[events['block']==blk].groupby('trial')['is_correct'].first().tolist()
    acc_str  = '  '.join(['✓' if a else '✗' for a in acc_vals])
    out_name = f'trial_detail_block{blk}.png'
    print(f"\nGenerating Block {blk} (span {span}) detail...")
    plot_trial_detail(blk, f'Span {span} — Trials: {acc_str}', out_name)

print("\nAll figures saved. Done.")

Loading data...
  Events    : 52 sub-trials, 7 blocks
  Neural    : 70,000 samples (Left, 250Hz)
  LFP power : 560 samples (2Hz)
  Spike events generated: 132

Generating Figure 1: Full session overview...
  ✓ session_overview.png saved → C:\Users\ASSUS\ATN\Digit Span Backwards\Plots\session_overview.png

Generating Block 1 (span 2) detail...
  ✓ trial_detail_block1.png saved → C:\Users\ASSUS\ATN\Digit Span Backwards\Plots\trial_detail_block1.png

Generating Block 2 (span 3) detail...
  ✓ trial_detail_block2.png saved → C:\Users\ASSUS\ATN\Digit Span Backwards\Plots\trial_detail_block2.png

Generating Block 3 (span 4) detail...
  ✓ trial_detail_block3.png saved → C:\Users\ASSUS\ATN\Digit Span Backwards\Plots\trial_detail_block3.png

Generating Block 4 (span 4) detail...
  ✓ trial_detail_block4.png saved → C:\Users\ASSUS\ATN\Digit Span Backwards\Plots\trial_detail_block4.png

Generating Block 5 (span 4) detail...
  ✓ trial_detail_block5.png saved → C:\Users\ASSUS\ATN\Digit Span Backwards